<a href="https://colab.research.google.com/github/sabascodes/Logistics-project/blob/main/transportationmodel_improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Barakah Logistics – Best Transport Mode (improved)
Changes: weight included, total cost used, one-hot encoding, cross-validation to check overfitting.

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving barakah_logistics_routes_las.csv to barakah_logistics_routes_las.csv


In [ ]:
df = pd.read_csv("barakah_logistics_routes_las.csv")
df.head()

,Origin,Destination,Distance_km,Season,Mode,Weight_kg,Base_Cost_SAR,Weight_Cost_SAR,Total_Cost_SAR,Delivery_Days,Delay_Probability
0,Jeddah,Mecca,90,Normal,Truck,500,30.0,20.0,50.0,1,0.10
1,Jeddah,Mecca,90,Normal,Truck,2000,30.0,100.0,130.0,1,0.12
2,Jeddah,Mecca,90,Normal,Truck,5000,30.0,250.0,280.0,1,0.15
3,Jeddah,Mecca,90,Normal,Air,500,380.0,120.0,500.0,1,0.05
4,Jeddah,Mecca,90,Normal,Air,2000,380.0,500.0,880.0,1,0.05


## 1. Label the best mode for each shipment
**Change:** `Weight_kg` is now part of what defines a shipment, so 500 kg and 5000 kg are compared separately.

In [ ]:
group_cols = ["Origin", "Destination", "Distance_km", "Season", "Weight_kg"]

def normalize(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)  # scale 0-1, +1e-9 avoids divide by zero

**Change:** use `Total_Cost_SAR` (includes the weight cost) instead of `Base_Cost_SAR`.

In [ ]:
df["cost_n"]  = df.groupby(group_cols)["Total_Cost_SAR"].transform(normalize)
df["time_n"]  = df.groupby(group_cols)["Delivery_Days"].transform(normalize)
df["delay_n"] = df.groupby(group_cols)["Delay_Probability"].transform(normalize)

W_COST, W_DELAY, W_TIME = 0.35, 0.25, 0.40
df["score"] = W_COST*df["cost_n"] + W_DELAY*df["delay_n"] + W_TIME*df["time_n"]

idx_best = df.groupby(group_cols)["score"].idxmin()
best = df.loc[idx_best, group_cols + ["Mode"]].rename(columns={"Mode": "Best_Mode"})

print(f"Number of unique route/season/weight combinations: {len(best)}")
print(best["Best_Mode"].value_counts())

Number of unique route/season/weight combinations: 54
Best_Mode
Train    27
Air      18
Truck     9
Name: count, dtype: int64


## 2. Features
**Change:** one-hot encoding instead of LabelEncoder, so city names aren't treated as ranked numbers.

In [ ]:
features = group_cols
X = pd.get_dummies(best[features], columns=["Origin", "Destination", "Season"], dtype=int)
y = best["Best_Mode"]
print("Feature columns:", list(X.columns))

Feature columns: ['Distance_km', 'Weight_kg', 'Origin_Jeddah', 'Origin_Riyadh', 'Destination_Abha', 'Destination_Dammam', 'Destination_Mecca', 'Destination_Medina', 'Destination_Riyadh', 'Season_Hajj', 'Season_Normal', 'Season_Ramadan']


In [ ]:
# stratify keeps the same mix of modes in train and test (needs >= 2 examples per mode)
strat = y if y.value_counts().min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=strat
)
print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")

Train rows: 37, Test rows: 17


## 3. Pick the tree depth with cross-validation
Instead of trusting one small test split, we test each depth on 5 different splits of the training data and average.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
for depth in [1, 2, 3, 4, 5, None]:
    m = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=2, random_state=42)
    scores = cross_val_score(m, X_train, y_train, cv=cv)
    results.append((depth, scores.mean(), scores.std()))
    print(f"max_depth={str(depth):>4}  CV accuracy = {scores.mean():.2%} (+/- {scores.std():.2%})")

# best mean score; on a tie, prefer the simpler (shallower) tree
best_depth = sorted(results, key=lambda r: (-round(r[1], 4), 99 if r[0] is None else r[0]))[0][0]
print(f"\nChosen max_depth: {best_depth}")

max_depth=   1  CV accuracy = 83.57% (+/- 6.12%)
max_depth=   2  CV accuracy = 100.00% (+/- 0.00%)
max_depth=   3  CV accuracy = 100.00% (+/- 0.00%)
max_depth=   4  CV accuracy = 100.00% (+/- 0.00%)
max_depth=   5  CV accuracy = 100.00% (+/- 0.00%)
max_depth=None  CV accuracy = 100.00% (+/- 0.00%)

Chosen max_depth: 2


## 4. Final model + overfitting check
If train accuracy is much higher than test accuracy, the model is overfitting.

In [ ]:
model = DecisionTreeClassifier(max_depth=best_depth, min_samples_leaf=2, random_state=42)
model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc  = accuracy_score(y_test, model.predict(X_test))
print(f"Train accuracy: {train_acc:.2%}")
print(f"Test accuracy:  {test_acc:.2%}")
print(f"Gap:            {train_acc - test_acc:.2%}  (small gap = not overfitting)\n")
print(classification_report(y_test, model.predict(X_test), zero_division=0))

Train accuracy: 100.00%
Test accuracy:  100.00%
Gap:            0.00%  (small gap = not overfitting)

              precision    recall  f1-score   support

         Air       1.00      1.00      1.00         6
       Train       1.00      1.00      1.00         8
       Truck       1.00      1.00      1.00         3

    accuracy                           1.00        17
   macro avg       1.00      1.00      1.00        17
weighted avg       1.00      1.00      1.00        17



In [ ]:
print("Decision tree rules:")
print(export_text(model, feature_names=list(X.columns)))

Decision tree rules:
|--- Distance_km <= 535.00
|   |--- class: Train
|--- Distance_km >  535.00
|   |--- Destination_Abha <= 0.50
|   |   |--- class: Air
|   |--- Destination_Abha >  0.50
|   |   |--- class: Truck



## 5. Predictions
A helper that one-hot encodes a new shipment the same way as the training data.

In [ ]:
def predict_mode(origin, destination, distance_km, season, weight_kg):
    row = pd.DataFrame([{"Origin": origin, "Destination": destination,
                         "Distance_km": distance_km, "Season": season, "Weight_kg": weight_kg}])
    row = pd.get_dummies(row, columns=["Origin", "Destination", "Season"], dtype=int)
    row = row.reindex(columns=X.columns, fill_value=0)  # match training columns
    return model.predict(row)[0]

for w in [500, 2000, 5000]:
    print(f"Jeddah -> Mecca,  Hajj, {w}kg: {predict_mode('Jeddah', 'Mecca', 90, 'Hajj', w)}")
for w in [500, 2000, 5000]:
    print(f"Jeddah -> Riyadh, Hajj, {w}kg: {predict_mode('Jeddah', 'Riyadh', 950, 'Hajj', w)}")

Jeddah -> Mecca,  Hajj, 500kg: Train
Jeddah -> Mecca,  Hajj, 2000kg: Train
Jeddah -> Mecca,  Hajj, 5000kg: Train
Jeddah -> Riyadh, Hajj, 500kg: Air
Jeddah -> Riyadh, Hajj, 2000kg: Air
Jeddah -> Riyadh, Hajj, 5000kg: Air
